# Interactive harmonic validation

[`validation.md`](validation.md) shows one frozen comparison: 163.8 ps of NVE argon
against the harmonic reference, at one resolution, from one random seed. Every number in
it is the result of choices — how long to run, how finely to resolve, which draw of
initial velocities to take — and a single figure cannot show whether those choices
matter. This notebook moves them.

It is built on the same harness as the static figures (`tests/lj_reference.py`), so a
number read off a slider here is the number the M2 test suite would assert on.

**The headline result is not the one the static figure suggests.** Running the MD for
longer does *not* walk the first moment onto the harmonic value. It walks it onto a
*seed-dependent* value, and the scatter between seeds is several times the uncertainty
`validation.md` quotes. The convergence panel below shows that directly, and
[the section after it](#why-longer-is-not-better) explains why it has to be that way in
NVE.

## Running this notebook

Nothing here runs molecular dynamics. A single run is about 25 s, which is far too slow
to sit inside a slider callback, so the whole grid — ten seeds × eleven trajectory
lengths × five segment lengths, plus window, overlap, temperature and the mixed-species
crystal — is precomputed once into a cache and the controls only index into arrays:

```bash
uv sync --extra euphonic --group docs
uv run python docs/make_interactive_data.py          # ~6 minutes, cached afterwards
uv run python docs/make_interactive_data.py --refresh # force a rebuild
```

Two of the axes cost nothing. The MD is run once at full length and the velocities are
kept, so a "shorter run" is the first *N* frames of the same trajectory — exactly what a
shorter run of that seed would have produced. Segment length and overlap are properties
of the estimator rather than of the dynamics, so they too are re-derived from the stored
velocities. Only the seed needs its own run, and the seeds are the ten that
`tests/lj_reference.py` now uses for its own ensemble. In the run that produced this
cache, 550 Welch estimates took 5.4 s against 342 s of molecular dynamics.

[Section 9](#9-a-real-trajectory-solid-benzene-from-mace-mp) adds a second, independent
cache built from a real MACE-MP trajectory of solid benzene. That one runs no dynamics
at all — the trajectory is on disk — so it is a fraction of a second, and it lives in
its own `.npz` precisely so that rebuilding it does not drag six minutes of
Lennard-Jones MD along with it. It needs
`docs/data/benzene_solid-average-nve-T25.0-traj.extxyz`, which is 12.9 MB and therefore
not committed; without it the first eight sections still build and the ninth raises a
`FileNotFoundError` naming the path.

The figures are Plotly, with their controls baked into the figure rather than driven by
callbacks, so the stored outputs stay interactive without a running kernel. GitHub's
notebook viewer strips JavaScript and will show nothing; use
[nbviewer](https://nbviewer.org/) or a local Jupyter.

In [1]:
# Load the precomputed grid and set up the shared plotting vocabulary.
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

ROOT = Path.cwd().parent if Path.cwd().name == "docs" else Path.cwd()
sys.path.insert(0, str(ROOT / "docs"))
sys.path.insert(0, str(ROOT))

# Broadening the harmonic reference is subtle enough that it is worth having exactly one
# implementation: this is the same function, and the same 1.44 Hann main-lobe factor,
# that produces the static figures in validation.md.
from make_interactive_data import load  # noqa: E402
from make_validation_figures import (  # noqa: E402
    HANN_MAIN_LOBE,
    broadened,
    density_curve,
)
from tests.lj_reference import moment, moment_uncertainty  # noqa: E402

# Renders in JupyterLab and, via the CDN copy of plotly.js, in nbviewer with no kernel.
pio.renderers.default = "plotly_mimetype+notebook_connected"

data = load()

ENERGIES = data["energies"]
SEEDS = data["seeds"].astype(int)
LENGTHS = data["lengths"].astype(int)
SEGMENTS = data["segments"].astype(int)
OVERLAPS = data["overlaps"]
OVERLAP_SEGMENTS = data["overlap_segments"].astype(int)
DT = float(data["dt"])
PICOSECONDS = LENGTHS * DT
MODES = data["single_modes"]

#: The harmonic answer the MD is being asked to reproduce: an unweighted mode list, so
#: its moments are plain means over the 78 finite modes of the 3x3x3 cell.
REFERENCE_M1 = float(MODES.mean())
REFERENCE_M2 = float((MODES**2).mean())

#: Okabe-Ito, distinguishable under all three common forms of colour blindness. Used for
#: the four windows, which are genuine categories; ten seeds are *not* ten categories —
#: they are ten draws from one distribution, so they share a single recessive colour and
#: the eye is pointed at the ensemble instead.
CATEGORY = ["#0072B2", "#D55E00", "#009E73", "#CC79A7"]
SEED_INK = "#98a2ad"
ENSEMBLE = "#0072B2"
#: Kept the same as the static figures so the two can be read side by side.
ARGON, KRYPTON, REFERENCE = "#1f77b4", "#d62728", "#555555"

#: Plotted range. The band tops out below 8 meV; carrying the full 0-20 meV estimation
#: grid into every trace would quadruple the size of this notebook for empty axis.
VIEW = ENERGIES <= 10.0


def harmonic_curve(resolution: float, lobe: float = HANN_MAIN_LOBE) -> np.ndarray:
    """The harmonic mode list as a curve, broadened by the estimator's own kernel.

    ``broadened`` assumes the Hann main lobe, so a different window is expressed as a
    rescaled resolution: what the reference should be convolved with is the main lobe of
    whichever window the estimate used.

    Args:
        resolution: Reported ``metadata.energy_resolution``, in meV.
        lobe: Main-lobe FWHM of the window, in bins of that reported resolution.

    Returns:
        Unit-area curve on ``ENERGIES``.
    """
    return broadened(
        MODES, np.ones_like(MODES), ENERGIES, resolution * lobe / HANN_MAIN_LOBE
    )


LAYOUT = {
    "template": "plotly_white",
    "font": {"size": 12},
    "margin": {"l": 70, "r": 30, "t": 80, "b": 60},
    "hovermode": "x unified",
}

print(f"grid: {len(SEEDS)} seeds x {len(LENGTHS)} lengths x {len(SEGMENTS)} segments")
print(f"trajectory lengths {PICOSECONDS[0]:.1f}-{PICOSECONDS[-1]:.1f} ps")
print(f"resolutions {data['grid_resolution'][0, -1]}")
print(f"harmonic reference: <E> = {REFERENCE_M1:.4f} meV, <E2> = {REFERENCE_M2:.3f}")

grid: 10 seeds x 11 lengths x 5 segments
trajectory lengths 10.2-163.8 ps
resolutions [0.8077476  0.4038738  0.2019369  0.10096845 0.05048422]
harmonic reference: <E> = 5.5852 meV, <E2> = 33.592


## 1. The spectrum as the run gets longer

The slider is the trajectory length, from 20 ps to the full 164 ps, at fixed 0.20 meV
resolution — so only the number of averaged Welch segments changes, from one to fifteen.
All ten seeds are drawn at once against the same broadened harmonic reference. They are
ten draws from one distribution rather than ten things worth telling apart, so they are
drawn in one recessive grey, with the two extremes picked out.

What to look for: the *noise* falls visibly as the segments accumulate, and that is the
part the Welch error bar measures. The *distribution of weight between the three clusters*
does not. Seed 7 puts 66% of its weight above 6 meV; seed 5 puts 39%. That 27-point gap is
there at 20 ps and still there at 164 ps. It is not noise, and no amount of extra sampling
removes it.

In [2]:
def spectrum_curves(segment_index: int) -> np.ndarray:
    """Unit-area pDOS for every (seed, length) at one resolution, over the view."""
    weights = data["grid_weight"][:, :, segment_index, :]
    areas = np.trapezoid(weights, ENERGIES, axis=-1)
    return (weights / areas[..., None])[..., VIEW]


SEGMENT = int(np.searchsorted(SEGMENTS, 1024))
curves = spectrum_curves(SEGMENT)
valid = [i for i, n in enumerate(LENGTHS) if n >= SEGMENTS[SEGMENT]]
resolution = float(data["grid_resolution"][0, -1, SEGMENT])

figure = go.Figure()
figure.add_trace(
    go.Scatter(
        x=ENERGIES[VIEW],
        y=harmonic_curve(resolution)[VIEW],
        name=f"harmonic, broadened to {HANN_MAIN_LOBE * resolution:.2f} meV",
        fill="tozeroy",
        line={"color": REFERENCE, "width": 0},
        fillcolor="rgba(85,85,85,0.22)",
        hoverinfo="skip",
    )
)
# The two extremes, by how much weight each seed puts in the upper cluster. Highlighting
# them is the whole argument of this panel: the envelope is a distribution, not noise.
upper = curves[:, -1][:, ENERGIES[VIEW] > 6.0].sum(axis=1)
extremes = {int(upper.argmin()): CATEGORY[0], int(upper.argmax()): CATEGORY[1]}

for length_index in valid:
    for seed_index, seed in enumerate(SEEDS):
        highlight = seed_index in extremes
        figure.add_trace(
            go.Scatter(
                x=ENERGIES[VIEW],
                y=np.round(curves[seed_index, length_index], 6),
                name=(
                    f"seed {seed}"
                    + (", lowest" if seed_index == upper.argmin() else "")
                    + (", highest" if seed_index == upper.argmax() else "")
                ),
                line={
                    "color": extremes.get(seed_index, SEED_INK),
                    "width": 2.0 if highlight else 1.1,
                },
                opacity=1.0 if highlight else 0.55,
                visible=length_index == valid[-1],
                legendgroup=f"seed{seed}",
                showlegend=highlight and length_index == valid[-1],
            )
        )

steps = []
for position, length_index in enumerate(valid):
    visible = [True] + [block == position for block in range(len(valid)) for _ in SEEDS]
    steps.append(
        {
            "label": f"{PICOSECONDS[length_index]:.0f}",
            "method": "update",
            "args": [
                {
                    "visible": visible,
                    "showlegend": [
                        shown and (index == 0 or (index - 1) % len(SEEDS) in extremes)
                        for index, shown in enumerate(visible)
                    ],
                },
                {
                    "title.text": (
                        f"FCC argon, 27 atoms, first "
                        f"{PICOSECONDS[length_index]:.1f} ps ("
                        f"{int(data['grid_n_segments'][0, length_index, SEGMENT])} "
                        f"Welch segments of {SEGMENTS[SEGMENT]} frames)"
                    )
                },
            ],
        }
    )

figure.update_layout(
    **LAYOUT,
    title=steps[-1]["args"][1]["title.text"],
    xaxis_title="energy transfer (meV)",
    yaxis_title="pDOS (1/meV, unit area)",
    sliders=[
        {
            "active": len(steps) - 1,
            "currentvalue": {"prefix": "trajectory length: ", "suffix": " ps"},
            "pad": {"t": 50},
            "steps": steps,
        }
    ],
    legend={"orientation": "h", "y": -0.32},
    height=520,
)
figure

## 2. Convergence of the moments — the panel that matters

The two moments the M2 tests assert on, against trajectory length, one line per seed.
Error bars are `moment_uncertainty`: the per-bin Welch inter-segment spread propagated
through the moment, which is exactly the tolerance `validation.md` derives its 3σ
threshold from. The dashed line is the harmonic answer.

The dropdown changes the Welch segment length, i.e. the resolution of the underlying
estimate. It is there to answer the obvious objection — that the pattern below is an
artefact of the estimator rather than of the dynamics. It is not: the curves barely move
across a factor of sixteen in resolution.

Read the panel this way:

- Each seed *does* settle: between 82 ps and 164 ps every ⟨E⟩ moves by less than 0.7%,
  and half of them by less than 0.1%.
- Each settles somewhere different, and mostly not on the dashed line — the ten span
  5.31 to 6.24 meV around a reference of 5.585 meV.
- The error bars shrink as √(number of segments) and end up several times smaller than
  the gaps between the seeds. The uncertainty being quoted is not the one that dominates.
- The heavy blue line is the ensemble mean with the standard error *of the mean*, taken
  over seeds. That is the statistic `tests/lj_reference.py` now exposes as
  `ensemble_moment`, and it is the honest one: it is flat in length from 20 ps onwards,
  because the length axis was never the problem.

In [3]:
def moment_traces(kind: str, segment_index: int) -> tuple[list, list]:
    """Per-seed moments, then the ensemble mean, with matching uncertainties.

    The per-seed error is the propagated Welch inter-segment spread; the ensemble error
    is the standard error of the mean over seeds, which is the sampling distribution of
    the statistic itself and needs no assumption about what is correlated with what.
    """
    values = data[f"grid_{kind}"][:, :, segment_index]
    errors = data[f"grid_s{kind[-1]}"][:, :, segment_index]
    mean = values.mean(axis=0)
    sem = values.std(axis=0, ddof=1) / np.sqrt(values.shape[0])
    return [*values, mean], [*errors, sem]


figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("first moment ⟨E⟩", "second moment ⟨E²⟩"),
    horizontal_spacing=0.09,
)
order: list[tuple[str, int]] = []  # trace order, for the dropdown's restyle

for column, (kind, reference) in enumerate(
    (("m1", REFERENCE_M1), ("m2", REFERENCE_M2)), start=1
):
    values, errors = moment_traces(kind, SEGMENT)
    for seed_index, seed in enumerate([*SEEDS, "mean"]):
        ensemble = seed == "mean"
        figure.add_trace(
            go.Scatter(
                x=PICOSECONDS,
                y=values[seed_index],
                error_y={
                    "type": "data",
                    "array": errors[seed_index],
                    "thickness": 1.6 if ensemble else 0.9,
                },
                mode="lines+markers" if ensemble else "lines",
                name=(
                    "ensemble mean ± SEM"
                    if ensemble
                    else ("individual seeds" if seed_index == 0 else f"seed {seed}")
                ),
                legendgroup="ensemble" if ensemble else "seeds",
                showlegend=column == 1 and (ensemble or seed_index == 0),
                marker={"size": 7},
                opacity=1.0 if ensemble else 0.6,
                line={
                    "color": ENSEMBLE if ensemble else SEED_INK,
                    "width": 3 if ensemble else 1.2,
                },
            ),
            row=1,
            col=column,
        )
        order.append((kind, seed_index))
    figure.add_trace(
        go.Scatter(
            x=PICOSECONDS,
            y=np.full_like(PICOSECONDS, reference),
            mode="lines",
            name="harmonic reference",
            legendgroup="reference",
            showlegend=column == 1,
            line={"color": REFERENCE, "width": 2, "dash": "dash"},
            hovertemplate=f"harmonic {reference:.4f}<extra></extra>",
        ),
        row=1,
        col=column,
    )
    order.append((kind, -1))

buttons = []
for segment_index, segment in enumerate(SEGMENTS):
    ys, es = [], []
    for kind, seed_index in order:
        reference = REFERENCE_M1 if kind == "m1" else REFERENCE_M2
        if seed_index < 0:
            ys.append(np.full_like(PICOSECONDS, reference))
            es.append(np.zeros_like(PICOSECONDS))
        else:
            values, errors = moment_traces(kind, segment_index)
            ys.append(values[seed_index])
            es.append(errors[seed_index])
    buttons.append(
        {
            "label": (
                f"{segment} frames \u2014 "
                f"{data['grid_resolution'][0, -1, segment_index]:.3f} meV"
            ),
            "method": "update",
            "args": [{"y": ys, "error_y.array": es}],
        }
    )

figure.update_layout(
    **{**LAYOUT, "hovermode": "closest"},
    title="Moments against trajectory length, ten independent NVE seeds",
    height=520,
    updatemenus=[
        {
            "buttons": buttons,
            "active": SEGMENT,
            "direction": "down",
            "x": 1.0,
            "xanchor": "right",
            "y": 1.16,
            "showactive": True,
        }
    ],
    annotations=[
        *figure.layout.annotations,
        {
            "text": "Welch segment:",
            "x": 0.63,
            "y": 1.14,
            "xref": "paper",
            "yref": "paper",
            "showarrow": False,
            "xanchor": "right",
        },
    ],
    legend={"orientation": "h", "y": -0.22},
)
figure.update_xaxes(title_text="trajectory length (ps)", type="log")
figure.update_yaxes(title_text="⟨E⟩ (meV)", row=1, col=1)
figure.update_yaxes(title_text="⟨E²⟩ (meV²)", row=1, col=2)
figure

<a id="why-longer-is-not-better"></a>
## 3. Why longer is not better: the frozen draw

This is the most important thing in the notebook, and it is a property of the physics,
not a defect of the code.

These runs are **NVE**. `MaxwellBoltzmannDistribution` draws one set of atomic
velocities, and the subsequent dynamics is very nearly harmonic — that is the whole
premise of the comparison. In a harmonic system the energy of every normal mode is a
constant of motion. The draw therefore fixes, once and for all, how the total energy is
distributed over the 78 modes, and the pDOS is a picture of exactly that distribution.

The consequence is uncomfortable: **averaging over more time cannot fix an unrepresentative
draw**, because there is nothing left to average over. A longer run measures the same
frozen mode energies more precisely. That is why the lines in panel 2 flatten rather than
converge, and it is why they flatten at different heights.

With 27 atoms and 78 modes the draw is coarse: fluctuations of order 1/√78 ≈ 11% in the
energy of any group of modes, and the first moment is a weighted comparison between the
high and low clusters, so a few percent of scatter is exactly what one should expect.

The numbers below make the size of the effect concrete.

In [4]:
segment_index = int(np.searchsorted(SEGMENTS, 2048))  # the setting validation.md uses
print(f"Welch segment {SEGMENTS[segment_index]} frames, full 163.8 ps, per seed:\n")
print("seed   <E> (meV)   quoted 1σ    (MD-ref)/ref   deviation")
for seed_index, seed in enumerate(SEEDS):
    value = data["grid_m1"][seed_index, -1, segment_index]
    sigma = data["grid_s1"][seed_index, -1, segment_index]
    print(
        f"  {seed}     {value:.4f}     ±{sigma:.4f}      "
        f"{(value - REFERENCE_M1) / REFERENCE_M1:+7.2%}     "
        f"{(value - REFERENCE_M1) / sigma:+5.1f}σ"
    )

values = data["grid_m1"][:, -1, segment_index]
sigmas = data["grid_s1"][:, -1, segment_index]
spread = values.std(ddof=1)
print(
    f"\nseed-to-seed scatter      {spread:.4f} meV   ({spread / values.mean():.2%})"
    f"\nmean quoted Welch 1σ      {sigmas.mean():.4f} meV   "
    f"({sigmas.mean() / values.mean():.2%})"
    f"\nratio                     {spread / sigmas.mean():.1f}x"
)

short = data["grid_m1"][:, LENGTHS.tolist().index(2048), segment_index]
print(
    f"\nsame scatter at 41 ps     {short.std(ddof=1) / short.mean():.2%}"
    f"  (vs {spread / values.mean():.2%} at 164 ps: 4x longer does not help)"
)

# The mean over seeds *is* an ensemble average, and its error shrinks as 1/sqrt(n).
sem = spread / np.sqrt(values.size)
print(
    f"\nensemble mean of {values.size} seeds  {values.mean():.4f} meV  ±{sem:.4f} (SEM)"
    f"\n  vs harmonic {REFERENCE_M1:.4f} meV: "
    f"{(values.mean() - REFERENCE_M1) / REFERENCE_M1:+.2%}, "
    f"{(values.mean() - REFERENCE_M1) / sem:+.1f} SEM"
)

Welch segment 2048 frames, full 163.8 ps, per seed:

seed   <E> (meV)   quoted 1σ    (MD-ref)/ref   deviation
  0     5.5525     ±0.0528       -0.58%      -0.6σ
  1     5.7128     ±0.0519       +2.29%      +2.5σ
  2     5.9289     ±0.0607       +6.15%      +5.7σ
  3     5.6853     ±0.0532       +1.79%      +1.9σ
  4     5.5242     ±0.0539       -1.09%      -1.1σ
  5     5.3092     ±0.0615       -4.94%      -4.5σ
  6     5.7059     ±0.0603       +2.16%      +2.0σ
  7     6.2427     ±0.0508      +11.77%     +13.0σ
  8     5.7034     ±0.0468       +2.12%      +2.5σ
  9     5.7593     ±0.0574       +3.12%      +3.0σ

seed-to-seed scatter      0.2483 meV   (4.35%)
mean quoted Welch 1σ      0.0549 meV   (0.96%)
ratio                     4.5x

same scatter at 41 ps     4.23%  (vs 4.35% at 164 ps: 4x longer does not help)

ensemble mean of 10 seeds  5.7124 meV  ±0.0785 (SEM)
  vs harmonic 5.5852 meV: +2.28%, +1.6 SEM


**What this means for `validation.md`.** The single-species entry in its results table
(⟨E⟩ = 5.5525 meV, −0.58%, −0.6σ) is reproduced exactly here — it is seed 0, and seed 0
happens to be one of the closest of the ten. The error bar beside it is real, but it
measures the wrong thing: it is the precision with which this run's frozen mode
distribution has been measured, not the accuracy with which that distribution represents
the thermodynamic ensemble. A test that draws a different seed can fail at 3σ without
anything being wrong with the pipeline.

Note the last line above. The ensemble mean sits **above** the harmonic reference, by
rather more than seed 0's individual deviation sits below it, and unlike the per-seed
scatter that offset does *not* shrink with more seeds — only the error on it does. Section
7 gives the reason: it is the anharmonic stiffening at finite temperature, which is
invisible in a single seed because the frozen-draw scatter is twice as large again.

The second moment is worse, because it weights the band edges hardest:

In [5]:
values = data["grid_m2"][:, -1, segment_index]
sigmas = data["grid_s2"][:, -1, segment_index]
print("seed   <E2> (meV2)   quoted 1σ     (MD-ref)/ref   deviation")
for seed_index, seed in enumerate(SEEDS):
    print(
        f"  {seed}     {values[seed_index]:7.3f}      ±{sigmas[seed_index]:.3f}       "
        f"{(values[seed_index] - REFERENCE_M2) / REFERENCE_M2:+7.2%}     "
        f"{(values[seed_index] - REFERENCE_M2) / sigmas[seed_index]:+5.1f}σ"
    )
print(
    f"\nseed-to-seed scatter {values.std(ddof=1) / values.mean():.2%}, "
    f"mean quoted 1σ {sigmas.mean() / values.mean():.2%}, "
    f"ratio {values.std(ddof=1) / sigmas.mean():.1f}x"
)

seed   <E2> (meV2)   quoted 1σ     (MD-ref)/ref   deviation
  0      33.436      ±0.580        -0.46%      -0.3σ
  1      34.845      ±0.594        +3.73%      +2.1σ
  2      37.852      ±0.664       +12.68%      +6.4σ
  3      35.307      ±0.577        +5.11%      +3.0σ
  4      33.303      ±0.587        -0.86%      -0.5σ
  5      31.107      ±0.644        -7.40%      -3.9σ
  6      35.296      ±0.663        +5.07%      +2.6σ
  7      41.145      ±0.568       +22.49%     +13.3σ
  8      35.185      ±0.527        +4.74%      +3.0σ
  9      35.896      ±0.626        +6.86%      +3.7σ

seed-to-seed scatter 7.68%, mean quoted 1σ 1.71%, ratio 4.5x


The honest fix is not a looser tolerance, and the repository has since taken the first of
the two routes below: `tests/lj_reference.py` now exposes `SEEDS`, `ensemble` and
`ensemble_moment`, and the cache behind this notebook uses the same ten seeds, so the
numbers here and the numbers in the test suite are the same numbers.

The fix is either to average several seeds (the scatter of the mean of *n* draws falls as
1/√n, because *those* are independent samples of the ensemble), or to sample the ensemble within one run with a decorrelating thermostat —
which the method review warns has its own cost, since a strongly coupled thermostat
perturbs the very dynamics being measured. Both are outside M2's scope; what matters here
is that the quoted uncertainty is a lower bound, and the notebook says so.

## 4. Resolution, and why the reference has to be broadened

The slider is the Welch segment length: longer segments buy resolution and pay for it in
averaging. Three curves move together:

- **MD** (blue), which sharpens as the segment lengthens and gets noisier as the number
  of segments falls.
- **The reference broadened by 1.44 × the reported bin spacing** (grey fill). The
  estimator's kernel is the Hann main lobe, which is 1.44 bins wide at half maximum, and
  `docs/make_validation_figures.py` explains why using the reported spacing directly is
  wrong.
- **The reference broadened by 1.00 × the spacing** (dotted), which is that mistake, drawn
  so you can see what it costs. It is a spike against a peak of finite height, and the eye
  reads the height difference as disagreement between two curves that in fact carry the
  same area.

At 256 frames (0.81 meV) the three clusters merge into one envelope and the comparison is
uninformative. At 4096 frames (0.05 meV) the resolution is finer than the spread of modes
within a cluster and only three segments remain to average, so the MD curve is visibly
ragged. The M2 choice of 2048 sits where the clusters are separated and fifteen segments
still contribute.

In [6]:
DEFAULT_SEGMENT = int(np.searchsorted(SEGMENTS, 2048))  # the M2 setting
weights = data["grid_weight"][0, -1]  # seed 0, full length, every segment length
resolutions = data["grid_resolution"][0, -1]

figure = go.Figure()
for segment_index in range(len(SEGMENTS)):
    visible = segment_index == DEFAULT_SEGMENT
    reference = broadened(
        MODES, np.ones_like(MODES), ENERGIES, float(resolutions[segment_index])
    )
    narrow = broadened(
        MODES,
        np.ones_like(MODES),
        ENERGIES,
        float(resolutions[segment_index]) / HANN_MAIN_LOBE,
    )
    figure.add_trace(
        go.Scatter(
            x=ENERGIES[VIEW],
            y=np.round(reference[VIEW], 6),
            name="harmonic, broadened by 1.44 bins (the Hann main lobe)",
            fill="tozeroy",
            line={"color": REFERENCE, "width": 0},
            fillcolor="rgba(85,85,85,0.22)",
            visible=visible,
            showlegend=visible,
        )
    )
    figure.add_trace(
        go.Scatter(
            x=ENERGIES[VIEW],
            y=np.round(narrow[VIEW], 6),
            name="harmonic, broadened by 1.00 bin (under-broadened)",
            line={"color": REFERENCE, "width": 1.4, "dash": "dot"},
            visible=visible,
            showlegend=visible,
        )
    )
    figure.add_trace(
        go.Scatter(
            x=ENERGIES[VIEW],
            y=np.round(density_curve(ENERGIES, weights[segment_index])[VIEW], 6),
            name="mdins, MD velocity pDOS",
            line={"color": ARGON, "width": 1.8},
            visible=visible,
            showlegend=visible,
        )
    )

steps = []
for segment_index, segment in enumerate(SEGMENTS):
    visible = [
        block == segment_index for block in range(len(SEGMENTS)) for _ in range(3)
    ]
    steps.append(
        {
            "label": f"{segment}",
            "method": "update",
            "args": [
                {"visible": visible, "showlegend": visible},
                {
                    "title.text": (
                        f"Seed 0, 163.8 ps, segment {segment} frames: "
                        f"{resolutions[segment_index]:.3f} meV bins, "
                        f"{int(data['grid_n_segments'][0, -1, segment_index])} "
                        f"segments averaged"
                    )
                },
            ],
        }
    )

figure.update_layout(
    **LAYOUT,
    title=steps[DEFAULT_SEGMENT]["args"][1]["title.text"],
    xaxis_title="energy transfer (meV)",
    yaxis_title="pDOS (1/meV, unit area)",
    sliders=[
        {
            "active": DEFAULT_SEGMENT,
            "currentvalue": {"prefix": "Welch segment: ", "suffix": " frames"},
            "pad": {"t": 50},
            "steps": steps,
        }
    ],
    legend={"orientation": "h", "y": -0.32},
    height=520,
)
figure

## 5. Windowing: the resolution that is reported and the resolution that is delivered

Segment length, window shape and overlap are not three unrelated knobs. They are the
three parameters of one operation — cutting the run into pieces, tapering each piece and
averaging the periodograms — and they trade the same two things against each other: how
finely the estimator can resolve, and how much of the true spectrum leaks to where it does
not belong.

`metadata.energy_resolution` reports only *h*/(*L*·d*t*), the bin spacing of one segment.
That number does not depend on the window at all. What the window sets is the *kernel*
the true spectrum is convolved with, and both of its features matter:

- the **main lobe**, whose width is the resolution actually delivered — 1.44 bins for
  Hann, and `docs/make_validation_figures.py` uses exactly that factor when it broadens
  the reference, because otherwise the reference is a spike against a peak of finite
  height;
- the **sidelobes**, which are the leakage floor. Hann's fall away steeply; the boxcar's,
  at −13 dB and decaying as 1/*f*, put weight far from the mode that produced it.

The left panel is the kernel itself, from `scipy.signal.get_window` — the same call
`mdins.spectral` makes. The right panel is what that does to the estimate, on a log axis
so the tails are visible, with the harmonic band top marked and the reference broadened by
*that window's* lobe rather than always by Hann's. The slider changes the window; **the
reported resolution does not change as you move it.**

Watch the boxcar. Its main lobe is the *narrowest* of the four — by the reported number it
would be the best-resolved of the estimates — and its tail above the band is the highest.
At the 2048-frame segment `validation.md` uses, the four windows agree on ⟨E⟩ to four
decimal places and on ⟨E²⟩ to within 0.4%; the visible difference is confined to the far
tail, where the boxcar's floor above 12 meV is about four times Hann's. That is a mild
penalty, and it is worth being clear that at this segment length the window choice is
nearly irrelevant to the reported numbers.

It stops being irrelevant when the segment is short, which is the regime a shorter
trajectory is forced into — and it is the tails, not the peaks, that pay. The cell after
the figure sweeps the segment length to show where the trade turns.

In [7]:
from scipy.signal import get_window  # the call mdins.spectral makes

WINDOWS = [str(name) for name in data["windows"]]
OVERSAMPLE = 64  # zero-padding factor, so the kernel is drawn as a curve, not a comb


def kernel(name: str, length: int = 1024) -> tuple[np.ndarray, np.ndarray]:
    """Power response of a window, in bins of the segment's own frequency spacing."""
    taper = get_window(name, length, fftbins=True)
    power = np.abs(np.fft.rfft(taper, n=length * OVERSAMPLE)) ** 2
    return np.arange(power.size) / OVERSAMPLE, power / power[0]


def main_lobe(name: str) -> float:
    """Full width at half maximum of the main lobe, in bins of the reported spacing."""
    offset, power = kernel(name)
    first = int(np.argmax(power < 0.5))
    pair = slice(first, first - 2, -1)  # ascending in power, for np.interp
    return 2 * float(np.interp(0.5, power[pair], offset[pair]))


LOBES = {name: main_lobe(name) for name in WINDOWS}
BAND_TOP = float(MODES.max())
resolution = float(data["window_resolution"][0, DEFAULT_SEGMENT])
outside = ENERGIES > BAND_TOP + 5 * resolution

print(f"Welch segment {SEGMENTS[DEFAULT_SEGMENT]} frames, seed 0, 163.8 ps.")
print("The reported resolution is h/(L·dt) and is the same on every row.\n")
print("window      main lobe  reported   delivered   leaked    <E>      <E2>")
print("            (bins)     (meV)      (meV)       weight    (meV)    (meV2)")
for index, name in enumerate(WINDOWS):
    weight = data["window_weight"][index, DEFAULT_SEGMENT]
    print(
        f"{name:<11} {LOBES[name]:.2f}       {resolution:.3f}      "
        f"{LOBES[name] * resolution:.3f}       "
        f"{weight[outside].sum() / weight.sum():6.2%}   "
        f"{data['window_m1'][index, DEFAULT_SEGMENT]:.4f}  "
        f"{data['window_m2'][index, DEFAULT_SEGMENT]:7.3f}"
    )
print(
    f"\nharmonic reference                                     "
    f"{REFERENCE_M1:.4f}  {REFERENCE_M2:7.3f}"
)

Welch segment 2048 frames, seed 0, 163.8 ps.
The reported resolution is h/(L·dt) and is the same on every row.

window      main lobe  reported   delivered   leaked    <E>      <E2>
            (bins)     (meV)      (meV)       weight    (meV)    (meV2)
hann        1.44       0.101      0.145        2.50%   5.5525   33.436
hamming     1.30       0.101      0.132        2.51%   5.5526   33.438
blackman    1.64       0.101      0.166        2.50%   5.5525   33.437
boxcar      0.89       0.101      0.089        3.01%   5.5556   33.576

harmonic reference                                     5.5852   33.592


In [8]:
figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "the kernel: window power response",
        "the consequence: estimated pDOS",
    ),
    horizontal_spacing=0.09,
)

# Left panel: all four kernels at once, because the comparison is the whole point.
for index, name in enumerate(WINDOWS):
    offset, power = kernel(name)
    keep = offset <= 8.0
    figure.add_trace(
        go.Scatter(
            x=np.round(offset[keep], 4),
            y=np.round(np.maximum(power[keep], 1e-8), 10),
            name=f"{name} ({LOBES[name]:.2f}-bin lobe)",
            line={"color": CATEGORY[index], "width": 1.8},
        ),
        row=1,
        col=1,
    )

for index, name in enumerate(WINDOWS):
    visible = name == "hann"
    weight = data["window_weight"][index, DEFAULT_SEGMENT]
    figure.add_trace(
        go.Scatter(
            x=ENERGIES[VIEW],
            y=np.round(harmonic_curve(resolution, lobe=LOBES[name])[VIEW], 8),
            name="harmonic, broadened by this window's own lobe",
            fill="tozeroy",
            line={"color": REFERENCE, "width": 0},
            fillcolor="rgba(85,85,85,0.22)",
            visible=visible,
            showlegend=visible,
        ),
        row=1,
        col=2,
    )
    figure.add_trace(
        go.Scatter(
            x=ENERGIES[VIEW],
            y=np.round(density_curve(ENERGIES, weight)[VIEW], 8),
            name=f"mdins, {name} window",
            line={"color": CATEGORY[index], "width": 1.8},
            visible=visible,
            showlegend=visible,
        ),
        row=1,
        col=2,
    )

steps = []
for index, name in enumerate(WINDOWS):
    visible = [True] * len(WINDOWS) + [
        block == index for block in range(len(WINDOWS)) for _ in range(2)
    ]
    steps.append(
        {
            "label": name,
            "method": "update",
            "args": [
                {"visible": visible, "showlegend": visible},
                {
                    "title.text": (
                        f"{name} window, segment {SEGMENTS[DEFAULT_SEGMENT]} frames: "
                        f"reported {resolution:.3f} meV, "
                        f"delivered {LOBES[name] * resolution:.3f} meV"
                    )
                },
            ],
        }
    )

figure.update_layout(
    **{**LAYOUT, "hovermode": "closest"},
    title=steps[0]["args"][1]["title.text"],
    height=560,
    sliders=[
        {
            "active": 0,
            "currentvalue": {"prefix": "window: "},
            "pad": {"t": 60},
            "steps": steps,
            "x": 0.55,
            "len": 0.45,
        }
    ],
    legend={"orientation": "h", "y": -0.45},
)
figure.add_vline(
    x=BAND_TOP,
    line={"color": REFERENCE, "width": 1, "dash": "dot"},
    annotation_text="band top",
    row=1,
    col=2,
)
figure.update_xaxes(title_text="offset from the true frequency (bins)", row=1, col=1)
figure.update_xaxes(title_text="energy transfer (meV)", range=[0, 10], row=1, col=2)
figure.update_yaxes(title_text="response", type="log", range=[-7, 0.2], row=1, col=1)
figure.update_yaxes(
    title_text="pDOS (1/meV, unit area)", type="log", range=[-5, 0.1], row=1, col=2
)
figure

In [9]:
print("Leakage above the band, and its cost in <E2>, as the segment shortens.")
print("'floor' is the median pDOS above 12 meV, where no mode exists at all.\n")
print("segment   window     leaked weight   <E2> (meV2)   error     floor")
far = ENERGIES > 12.0
for segment_index, segment in enumerate(SEGMENTS):
    for index, name in enumerate(WINDOWS):
        weight = data["window_weight"][index, segment_index]
        if not np.isfinite(weight).all():
            continue
        second = data["window_m2"][index, segment_index]
        density = density_curve(ENERGIES, weight)
        # The cut follows the segment's own resolution, as validation.md's does.
        band = ENERGIES > BAND_TOP + 5 * data["window_resolution"][index, segment_index]
        print(
            f"{segment:>6}    {name:<9}  {weight[band].sum() / weight.sum():8.2%}      "
            f"{second:8.3f}     {(second - REFERENCE_M2) / REFERENCE_M2:+6.2%}   "
            f"{np.median(density[far]):.2e}"
        )
    print()

Leakage above the band, and its cost in <E2>, as the segment shortens.
'floor' is the median pDOS above 12 meV, where no mode exists at all.

segment   window     leaked weight   <E2> (meV2)   error     floor
   256    hann          0.27%        33.870     +0.83%   7.23e-05
   256    hamming       0.28%        33.839     +0.73%   6.70e-05
   256    blackman      0.28%        33.945     +1.05%   8.69e-05
   256    boxcar        0.49%        34.345     +2.24%   2.48e-04

   512    hann          0.73%        33.530     -0.18%   2.64e-05
   512    hamming       0.74%        33.534     -0.17%   3.01e-05
   512    blackman      0.73%        33.534     -0.17%   2.56e-05
   512    boxcar        1.34%        34.183     +1.76%   2.51e-04

  1024    hann          1.93%        33.462     -0.39%   3.79e-05
  1024    hamming       1.95%        33.466     -0.37%   4.02e-05
  1024    blackman      1.91%        33.459     -0.39%   3.67e-05
  1024    boxcar        2.58%        33.826     +0.70%   1.47e-

Three things fall out of that table, and they are the reason the default is a Hann window
at 2048 frames rather than anything cheaper:

1. **The reported resolution never appears in it.** Every row at a given segment length
   reports the same `energy_resolution`, while the delivered width varies by a factor of
   almost two between boxcar and Blackman.
2. **Leakage is a short-segment problem.** Compare within a row, not down a column: the
   out-of-band cut is `band top + 5 × resolution`, so it moves outwards as the segment
   shortens and the raw percentages are not comparable between rows. Within a row, at
   256–512 frames the boxcar's noise floor is about ten times Hann's and it biases ⟨E²⟩
   by +1.8 to +2.2%; at 2048 frames the same penalty is 0.4%, and its out-of-band weight
   is 3.0% against 2.5%. Lengthening the segment suppresses leakage more effectively
   than choosing a better window does.
3. **Tapering is close to free.** Hann, Hamming and Blackman differ from one another by
   less than the seed-to-seed scatter of section 3 at every segment length. The decision
   that matters is *taper or not*; which taper is a second-order question.

So the window is not where the error budget lives — the frozen draw is, by an order of
magnitude. It earns its place in this notebook for a different reason: it is the one knob
where the number the code *reports* and the number the estimator *delivers* come apart,
and a reader who compares a 0.101 meV reported resolution against a peak that is really
0.145 meV wide will conclude the peak is anharmonically broadened when it is not.

## 6. Overlap, and what the error bar is actually counting

Overlap is the one knob that changes the *quoted uncertainty* far more than it changes
the *estimate*. Everything below is expressed as a percentage of ⟨E⟩, so it shares one
axis.

The bars are the quoted 1σ. At the M2 segment length of 2048 frames, raising the overlap
from 0 to 0.75 takes the segment count from 4 to 13 and the quoted error from 1.19% to
0.66% — while ⟨E⟩ itself moves by 0.004 meV, under a tenth of the error bar. Overlapping
segments share data and are correlated, so most of that shrinkage is bookkeeping;
`spectral.py` says as much in its docstring: *"use `overlap=0.0` where the error bar
matters."*

The two horizontal lines are the scale that the error bar should be compared against: the
distance from the harmonic answer, and the seed-to-seed scatter from panel 3. Every bar
sits below both. Whatever the overlap, the quoted uncertainty is smaller than the effect
it would need to cover.

In [10]:
figure = go.Figure()
for row, segment in enumerate(OVERLAP_SEGMENTS):
    percent = 100 * data["overlap_s1"][row] / data["overlap_m1"][row]
    figure.add_trace(
        go.Bar(
            x=[f"{value:.0%}" for value in OVERLAPS],
            y=np.round(percent, 3),
            name=f"segment {segment} frames",
            marker={
                "color": CATEGORY[row],
                "line": {"color": "white", "width": 2},
            },
            text=[
                f"{value:.2f}%<br>{int(n)} seg"
                for value, n in zip(
                    percent, data["overlap_n_segments"][row], strict=True
                )
            ],
            textposition="outside",
        )
    )

moments = data["grid_m1"][:, -1, DEFAULT_SEGMENT]
scatter = 100 * moments.std(ddof=1) / REFERENCE_M1
bias = abs(100 * (moments[0] - REFERENCE_M1) / REFERENCE_M1)
for value, label, colour in (
    (scatter, f"seed-to-seed scatter, {scatter:.1f}%", "#D55E00"),
    (bias, f"seed 0 distance from harmonic, {bias:.2f}%", REFERENCE),
):
    figure.add_hline(
        y=value,
        line={"color": colour, "width": 2, "dash": "dash"},
        annotation_text=label,
        annotation_position="top left",
    )

figure.update_layout(
    **{**LAYOUT, "hovermode": "x"},
    barmode="group",
    title="Quoted uncertainty on ⟨E⟩ against Welch overlap (seed 0, 163.8 ps)",
    xaxis_title="fractional overlap between segments",
    yaxis_title="as a percentage of ⟨E⟩",
    height=470,
    legend={"orientation": "h", "y": -0.22},
)
figure.update_yaxes(range=[0, max(scatter, 1.4) * 1.25])
figure

## 7. Temperature: where the harmonic premise starts to fail

`validation.md` argues the MD has to be cold enough to stay harmonic, and runs at about
9.3 K — roughly a fortieth of argon's melting point. This panel is that argument made
falsifiable. All three runs use seed 0, and `MaxwellBoltzmannDistribution` scales one draw
of normal deviates by √T, so the three start from the *same* frozen mode distribution at
three amplitudes: the seed effect of panel 3 is held fixed and what remains is
anharmonicity.

⟨E⟩ rises with temperature, by about 0.019 meV/K. At fixed volume that is the expected
direction for Lennard-Jones — larger excursions sample the steep repulsive wall, so the
effective force constants stiffen — and it is a real physical difference from the harmonic
reference, not an error. Going from 9.3 K to 18.7 K moves ⟨E⟩ by three times the quoted
Welch uncertainty. It is not merely leakage into the tail above the band: the same trend
survives when the moment is restricted to the band, and the weight outside the band grows
from 1.4% to 4.8% on top of it.

Because all three runs share one frozen draw, the temperature axis can be extrapolated,
and that is worth doing.

In [11]:
figure = go.Figure()
resolution = float(data["temperature_resolution"][0])
figure.add_trace(
    go.Scatter(
        x=ENERGIES[VIEW],
        y=np.round(harmonic_curve(resolution)[VIEW], 6),
        name="harmonic reference (temperature-independent)",
        fill="tozeroy",
        line={"color": REFERENCE, "width": 0},
        fillcolor="rgba(85,85,85,0.22)",
    )
)
shades = ["#9ecae1", ARGON, "#08306b"]
for index, temperature in enumerate(data["temperature_temperature"]):
    value = data["temperature_m1"][index]
    figure.add_trace(
        go.Scatter(
            x=ENERGIES[VIEW],
            y=np.round(
                density_curve(ENERGIES, data["temperature_weight"][index])[VIEW], 6
            ),
            name=(
                f"{temperature:.1f} K — ⟨E⟩ {value:.3f} meV "
                f"({(value - REFERENCE_M1) / REFERENCE_M1:+.1%})"
            ),
            line={"color": shades[index], "width": 1.8},
        )
    )

figure.update_layout(
    **LAYOUT,
    title="Same frozen draw, three temperatures: the harmonic premise under strain",
    xaxis_title="energy transfer (meV)",
    yaxis_title="pDOS (1/meV, unit area)",
    height=500,
    legend={"orientation": "h", "y": -0.3},
)
figure

In [12]:
band = ENERGIES <= MODES.max() + 5 * float(data["temperature_resolution"][0])
temperatures = data["temperature_temperature"]
full = data["temperature_m1"]
restricted = np.array(
    [moment(ENERGIES[band], w[band], 1) for w in data["temperature_weight"]]
)

print(" T (K)   <E> full range   <E> over the band   weight above the band")
for index, temperature in enumerate(temperatures):
    outside = data["temperature_weight"][index][~band].sum()
    print(
        f"{temperature:6.2f}   {full[index]:.4f} meV       {restricted[index]:.4f} meV"
        f"          {outside / data['temperature_weight'][index].sum():6.2%}"
    )

# Under a purely harmonic Hamiltonian the pDOS of a fixed draw is independent of its
# amplitude, so the T -> 0 intercept is what this seed's frozen mode distribution would
# give with no anharmonicity at all: the seed bias of panel 3, isolated.
for label, values in (("full range", full), ("over the band", restricted)):
    intercept = float(np.polyval(np.polyfit(temperatures, values, 1), 0.0))
    print(
        f"\nlinear T -> 0 extrapolation, {label}: {intercept:.3f} meV"
        f"  ({(intercept - REFERENCE_M1) / REFERENCE_M1:+.1%} vs harmonic)"
    )

 T (K)   <E> full range   <E> over the band   weight above the band
  4.62   5.4354 meV       5.3878 meV           1.37%
  9.27   5.5525 meV       5.4637 meV           2.50%
 18.65   5.7127 meV       5.5403 meV           4.80%

linear T -> 0 extrapolation, full range: 5.357 meV  (-4.1% vs harmonic)

linear T -> 0 extrapolation, over the band: 5.350 meV  (-4.2% vs harmonic)


**This is the most uncomfortable number in the notebook.** Extrapolated to zero
temperature, where seed 0's spectrum must become the harmonic spectrum of its own frozen
draw, ⟨E⟩ lands about 4% *below* the harmonic reference — whether or not the moment is
restricted to the band. At 9.3 K it is 0.6% below. In other words, seed 0's draw is biased
low and anharmonic stiffening at 9.3 K is biased high, and the −0.58% that
`validation.md` reports is the two partly cancelling.

Treat it as suggestive rather than established: it is three temperatures, one seed, and a
straight-line fit to a shift that need not be linear. But it says two things that do not
depend on the details. The first is that 9.3 K is not as deep in the harmonic regime as
"a fortieth of the melting point" makes it sound — the shift over 9 K is several times the
quoted error bar. The second is that a comparison agreeing to well inside 1σ is not, on
its own, evidence that both of its error sources are small.

## 8. Mixed species: which projection, and how long it takes to settle

The Ar/Kr crystal is the case the single-species comparison cannot address: when every
atom is equivalent, a projection error averages away. Here the two species feel identical
forces — ASE's Lennard-Jones takes one epsilon and sigma — and differ only by a mass ratio
of 2.10.

The slider is again trajectory length, and the two species do not settle at the same rate.
Both start about 3% high at 41 ps. Krypton, in the lower cluster where the modes are dense,
is inside 0.3% of the reference by 82 ps and stays there. Argon carries the top of the
band, where the modes are fewer and the draw is coarser, and its first moment is still
walking downwards at 143 ps — it crosses the reference somewhere around 100 ps and ends
0.6% below.

The 41 ps row has no error bar at all, and that is not a formatting failure: one segment
of 2048 frames fits into 2048 frames exactly once, and a single segment has no
inter-segment spread to report. The uncertainty `validation.md` quotes exists only because
there are several segments to disagree with each other.

Moments here are taken **over the band**, cut at the top of the harmonic spectrum plus
five resolution widths, which is the statistic `validation.md` quotes for this case: about
1% of the argon weight lies above the band top from Welch leakage and residual
anharmonicity, and ⟨E²⟩ weights that tail by energy squared. Recomputing from the stored
spectra rather than reading a cached number keeps that choice visible.

In [13]:
BINARY_SEGMENT = int(np.searchsorted(SEGMENTS, 2048))
resolution = float(data["binary_Ar_resolution"][-1, BINARY_SEGMENT])
band_top = max(data[f"binary_{s}_ref_energies"].max() for s in ("Ar", "Kr"))
inside = ENERGIES <= band_top + 5 * resolution

references, band_moments = {}, {}
for species in ("Ar", "Kr"):
    energies = data[f"binary_{species}_ref_energies"]
    weights = data[f"binary_{species}_ref_weights"]
    references[species] = broadened(energies, weights, ENERGIES, resolution)
    band_moments[species] = moment(energies, weights, 1)

valid = [i for i, n in enumerate(LENGTHS) if n >= SEGMENTS[BINARY_SEGMENT]]
table = []
for length_index in valid:
    row = [PICOSECONDS[length_index]]
    for species in ("Ar", "Kr"):
        weight = data[f"binary_{species}_weight"][length_index, BINARY_SEGMENT][inside]
        error = data[f"binary_{species}_error"][length_index, BINARY_SEGMENT][inside]
        row += [
            moment(ENERGIES[inside], weight, 1),
            moment_uncertainty(ENERGIES[inside], weight, error, 1),
        ]
    table.append(row)

print(
    "Ar/Kr first moments over the band (Euphonic: "
    f"Ar {band_moments['Ar']:.4f}, Kr {band_moments['Kr']:.4f} meV)\n"
)
print("  ps        Ar <E>              Kr <E>")
for ps, ar, ar_sigma, kr, kr_sigma in table:
    print(
        f"{ps:6.1f}   {ar:.4f} ±{ar_sigma:.4f} "
        f"({(ar - band_moments['Ar']) / band_moments['Ar']:+.2%})"
        f"   {kr:.4f} ±{kr_sigma:.4f} "
        f"({(kr - band_moments['Kr']) / band_moments['Kr']:+.2%})"
    )

Ar/Kr first moments over the band (Euphonic: Ar 5.5975, Kr 3.8407 meV)

  ps        Ar <E>              Kr <E>
  41.0   5.7788 ±nan (+3.24%)   3.9575 ±nan (+3.04%)
  61.4   5.6812 ±0.0382 (+1.49%)   3.9031 ±0.0282 (+1.63%)
  81.9   5.6327 ±0.0417 (+0.63%)   3.8504 ±0.0279 (+0.25%)
 102.4   5.6009 ±0.0371 (+0.06%)   3.8496 ±0.0239 (+0.23%)
 122.9   5.5769 ±0.0343 (-0.37%)   3.8530 ±0.0209 (+0.32%)
 143.4   5.5532 ±0.0314 (-0.79%)   3.8442 ±0.0197 (+0.09%)
 163.8   5.5638 ±0.0288 (-0.60%)   3.8461 ±0.0183 (+0.14%)


In [14]:
figure = go.Figure()
for species, colour in (("Ar", ARGON), ("Kr", KRYPTON)):
    figure.add_trace(
        go.Scatter(
            x=ENERGIES[VIEW],
            y=np.round(density_curve(ENERGIES, references[species])[VIEW], 6),
            name=f"{species}, Euphonic (broadened)",
            line={"color": colour, "width": 1.3, "dash": "dash"},
        )
    )
for length_index in valid:
    for species, colour in (("Ar", ARGON), ("Kr", KRYPTON)):
        weight = data[f"binary_{species}_weight"][length_index, BINARY_SEGMENT]
        figure.add_trace(
            go.Scatter(
                x=ENERGIES[VIEW],
                y=np.round(density_curve(ENERGIES, weight)[VIEW], 6),
                name=f"{species}, MD",
                line={"color": colour, "width": 1.9},
                visible=length_index == valid[-1],
                showlegend=length_index == valid[-1],
                legendgroup=species,
            )
        )

steps = []
for position, length_index in enumerate(valid):
    visible = [True, True] + [
        block == position for block in range(len(valid)) for _ in range(2)
    ]
    steps.append(
        {
            "label": f"{PICOSECONDS[length_index]:.0f}",
            "method": "update",
            "args": [
                {"visible": visible, "showlegend": visible},
                {
                    "title.text": (
                        "Ordered Ar/Kr, 32 atoms — identical forces, mass ratio 2.10, "
                        f"first {PICOSECONDS[length_index]:.1f} ps"
                    )
                },
            ],
        }
    )

figure.update_layout(
    **LAYOUT,
    title=steps[-1]["args"][1]["title.text"],
    xaxis_title="energy transfer (meV)",
    yaxis_title="pDOS (1/meV, unit area per species)",
    sliders=[
        {
            "active": len(steps) - 1,
            "currentvalue": {"prefix": "trajectory length: ", "suffix": " ps"},
            "pad": {"t": 50},
            "steps": steps,
        }
    ],
    legend={"orientation": "h", "y": -0.32},
    height=520,
)
figure

## 9. A real trajectory: solid benzene from MACE-MP

Everything above is a Lennard-Jones crystal generated inside this repository and
compared against a Euphonic reference computed from the same potential. That is the only
setting in which the word *validation* is honest. This section is the other case: a
trajectory produced elsewhere, with **no harmonic reference in this repository**, so
nothing below is a validation of anything. It is a demonstration that the stage-A/B
pipeline runs end to end on real data, and — as it turns out — that the estimator is a
sharper diagnostic of the *trajectory* than of itself.

The file is `docs/data/benzene_solid-average-nve-T25.0-traj.extxyz`: 1001 frames of an
NVE run on a 48-atom cell of solid benzene (four C₆H₆ molecules), forces from a
fine-tuned MACE-MP model, nominal temperature 25 K. It is 12.9 MB and is **not
committed**; the cache built from it is what the figures below index into.

Three things are different from the Lennard-Jones sections, and none of them can be
papered over:

1. **There is one trajectory, so there is no error bar.** The central result of this
   notebook — [§3](#why-longer-is-not-better) — is that in NVE the normal-mode
   energies are frozen by the initial velocity draw, so the Welch inter-segment spread
   understates the true run-to-run scatter by about 4.5×. That argument applies here
   with full force, and there is no second seed with which to measure the factor. Every
   ±1σ quoted below is therefore a *precision* figure for this one run and not an
   uncertainty on the answer.
2. **There is no reference curve.** No Euphonic force constants for benzene exist in
   this repository, so the grey filled band that anchors every earlier figure is simply
   absent.
3. **The physics is on a different energy scale.** Benzene's external (lattice and
   librational) modes sit below about 15 meV, but its intramolecular modes run from
   ≈ 49 meV (the ν₁₆ ring bend, 398 cm⁻¹) to ≈ 380 meV (C–H stretch). Whether the
   trajectory can see any of that is the first question, and it has a disappointing
   answer.


In [15]:
from make_interactive_data import load_benzene

from mdins.units import KB

bz = load_benzene()

BZ_E = bz["benzene_energies"]
BZ_LOW_E = bz["benzene_low_energies"]
BZ_LENGTHS = bz["benzene_lengths"].astype(int)
BZ_SEGMENTS = bz["benzene_segments"].astype(int)
BZ_WINDOWS = [str(name) for name in bz["benzene_windows"]]
BZ_OVERLAPS = bz["benzene_overlaps"]
BZ_DT = float(bz["benzene_dt"])
BZ_PS = BZ_LENGTHS * BZ_DT
BZ_NYQUIST = float(bz["benzene_nyquist"])
BZ_SPECIES = ("C", "H")

#: Carbon grey against hydrogen blue, kept distinct from the argon/krypton pair above so
#: that a reader scrolling between sections is not invited to compare them.
CARBON, HYDROGEN = "#7f7f7f", "#0072B2"

n_atoms = int(bz["benzene_n_atoms"])
n_molecules = int(bz["benzene_counts"][1]) // 6
external_dof = 6 * n_molecules
internal_dof = 3 * n_atoms - external_dof

print(
    f"{n_atoms} atoms, {n_molecules} benzene molecules, "
    f"{bz['benzene_n_frames']:.0f} frames"
)
print(
    f"dt          {BZ_DT * 1e3:.1f} fs  (read from the file: info['time'] in fs, "
    f"step 0/100/200 ...)"
)
print(f"duration    {float(bz['benzene_duration']):.1f} ps")
print(f"Nyquist     {BZ_NYQUIST:.3f} meV   <- the number that decides this section")
print(
    f"resolution  {4.135667696923859 / float(bz['benzene_duration']):.4f} meV "
    f"floor from the full duration"
)
print(
    f"kinetic T   {float(bz['benzene_temperature']):.2f} K, drifting "
    f"{100 * float(bz['benzene_temperature_drift']):.2f}% between the first and last "
    f"third"
)
print()
print("degrees of freedom per cell")
print(
    f"  external (3 translation + 3 libration per molecule)  {external_dof:3d}"
    f"   below ~15 meV"
)
print(
    f"  intramolecular                                       {internal_dof:3d}"
    f"   >= 49 meV, i.e. above Nyquist"
)
print(
    f"  fraction of the kinetic energy above Nyquist         "
    f"{internal_dof / (3 * n_atoms):.1%}"
)

48 atoms, 4 benzene molecules, 1001 frames
dt          100.0 fs  (read from the file: info['time'] in fs, step 0/100/200 ...)
duration    100.1 ps
Nyquist     20.678 meV   <- the number that decides this section
resolution  0.0413 meV floor from the full duration
kinetic T   20.21 K, drifting 0.10% between the first and last third

degrees of freedom per cell
  external (3 translation + 3 libration per molecule)   24   below ~15 meV
  intramolecular                                       120   >= 49 meV, i.e. above Nyquist
  fraction of the kinetic energy above Nyquist         83.3%


### 9.1 The frame spacing decides everything, and it was not chosen for this

`dt` is **not** a guess. Every frame's `info` carries `time` and `step` together with a
`units` dictionary that states `"time": "fs"`, and the frames run `step` 0, 100,
200 … at `time` 0.0, 100.0, 200.0 fs — a 1 fs integration timestep dumped every 100
steps. `make_interactive_data.benzene_dt` re-derives that from the file at build time and
raises if the spacing is non-uniform or disagrees with the `BENZENE_DT = 0.1` constant,
because every energy in this section scales inversely with it.

A 0.1 ps dump interval puts the Nyquist limit at **20.68 meV**. Benzene's lowest
intramolecular mode is at about 49 meV. So:

- the external modes, 24 of the cell's 144 degrees of freedom, are resolved;
- the other 120 — **83% of the kinetic energy** — are above the limit and do not
  simply go missing. They fold back. The ν₁₆ ring bend at 49 meV aliases to
  `|49 - 2 x 20.68| = 7.6 meV`, straight into the middle of the lattice band.

`VelocityTrajectory.validate_sampling` does refuse an `e_max` above the limit, which is
why `velocity_spectral_density` is called here with a grid that stops just short of it.
What it cannot do is warn about power that was already above the limit when the file was
written. Aliasing is irreversible after the fact, which is exactly why the package
raises rather than warns.

The next panel measures the folding instead of asserting it. Throw away every second
frame: `dt` becomes 0.2 ps, the Nyquist limit halves to 10.33 meV, and any weight that
appears below 10.33 meV that was not there before is weight that folded down. It is a
lower bound on the aliasing already present at 0.1 ps, obtained without any reference.


In [16]:
figure = go.Figure()

full = bz["benzene_grid_H"][-1, -1]  # full length, longest segment, dt = 0.1 ps
decimated = bz["benzene_decimated_H"][1]  # dt = 0.2 ps
low_top = float(BZ_LOW_E[-1])

figure.add_trace(
    go.Scatter(
        x=BZ_E,
        y=np.round(full, 6),
        name="dt = 0.1 ps, as written (Nyquist 20.68 meV)",
        line={"color": HYDROGEN, "width": 1.8},
    )
)
figure.add_trace(
    go.Scatter(
        x=BZ_LOW_E,
        y=np.round(decimated, 6),
        name="dt = 0.2 ps, every other frame (Nyquist 10.33 meV)",
        line={"color": CATEGORY[1], "width": 1.8, "dash": "dot"},
    )
)
figure.add_vline(
    x=low_top,
    line={"color": "#999999", "width": 1, "dash": "dash"},
    annotation_text="10.33 meV",
    annotation_position="top",
)

integrals = bz["benzene_decimated_H_integral"]
figure.update_layout(
    **LAYOUT,
    title=(
        "Hydrogen pDOS, per atom, absolute units. Halving the sampling rate adds "
        f"{integrals[1] / integrals[0] - 1:.0%} to the weight below 10.33 meV"
    ),
    xaxis_title="energy transfer (meV)",
    yaxis_title="tr P / 3 per H atom (Å² ps⁻² meV⁻¹)",
    legend={"orientation": "h", "y": -0.28},
    height=480,
)
figure

In [17]:
print("Weight below 10.33 meV, per atom, integrated tr P dE in (\u00c5/ps)\u00b2.")
print("If the trajectory were adequately sampled these two rows would agree.\n")
print("species   dt = 0.1 ps   dt = 0.2 ps   folded in   fraction of the full")
print("                                                  0-20.68 meV band")
for species in BZ_SPECIES:
    low = bz[f"benzene_decimated_{species}_integral"]
    band = float(bz[f"benzene_grid_{species}_integral"][-1, -1])
    print(
        f"   {species}      {low[0]:9.4f}     {low[1]:9.4f}     "
        f"{low[1] / low[0] - 1:+7.0%}        {low[0] / band:.1%}"
    )
print()
print(
    f"The 0-20.68 meV spectrum has not decayed at its own upper edge either: the mean "
    f"of the last ten bins is {full[-10:].mean() / full.mean():.0%} of the band mean. "
    "A properly sampled\nspectrum goes to zero at Nyquist; this one is still carrying "
    "aliased intramolecular weight when it runs out of axis."
)

Weight below 10.33 meV, per atom, integrated tr P dE in (Å/ps)².
If the trajectory were adequately sampled these two rows would agree.

species   dt = 0.1 ps   dt = 0.2 ps   folded in   fraction of the full
                                                  0-20.68 meV band
   C         1.7249        3.2884        +91%        52.6%
   H        35.2634       58.2683        +65%        60.1%

The 0-20.68 meV spectrum has not decayed at its own upper edge either: the mean of the last ten bins is 21% of the band mean. A properly sampled
spectrum goes to zero at Nyquist; this one is still carrying aliased intramolecular weight when it runs out of axis.


### 9.2 The sum rule catches something the aliasing cannot hide

Aliasing is a rearrangement, not a loss: folding conserves total power, so
`∫ tr Pᵢ dE` still equals `⟨|vᵢ|²⟩` even when every feature is in the wrong place.
That makes the equipartition sum rule, `∫ tr Pᵢ dE = 3 k_B T / mᵢ`, still meaningful
here — and it fails, by a lot, in a very specific way.

`VelocitySpectralDensity.check_sum_rule()` raises on this trajectory for 40 of the 48
atoms. The reason is not the estimator. Carbon and hydrogen are simply not at the same
kinetic temperature, and they stay that way for the whole 100 ps.


In [18]:
def parseval(species: str) -> float:
    """Welch integral against the direct mean-square velocity, as a percentage."""
    integral = float(bz[f"benzene_grid_{species}_integral"][-1, 2])
    return 100 * (integral / float(bz[f"benzene_msv_{species}"]) - 1)


system_t = float(bz["benzene_temperature"])
print(f"system kinetic temperature (3N-3 dof, COM removed): {system_t:.2f} K")
print("the file's own frame-0 `temperature` info field:     25.22 K\n")
print(
    "species  atoms   mass    <|v|\u00b2> direct   \u222b tr P dE (Welch)   "
    "3k_BT_sys/m   T_species"
)
for index, species in enumerate(BZ_SPECIES):
    mass = float(bz["benzene_masses"][index])
    direct = float(bz[f"benzene_msv_{species}"])
    welch = float(bz[f"benzene_grid_{species}_integral"][-1, 2])
    expected = 3 * KB * system_t / mass
    print(
        f"   {species}     {int(bz['benzene_counts'][index]):3d}   {mass:6.3f}   "
        f"{direct:11.4f}   {welch:14.4f}   {expected:11.4f}   "
        f"{float(bz[f'benzene_T_{species}']):7.2f} K"
    )

ratio = float(bz["benzene_T_H"]) / float(bz["benzene_T_C"])
mass_ratio = float(bz["benzene_masses"][0] / bz["benzene_masses"][1])
measured_ratio = float(
    bz["benzene_grid_H_integral"][-1, 2] / bz["benzene_grid_C_integral"][-1, 2]
)
print()
print(f"T_H / T_C                          {ratio:.3f}")
print(f"\u222b_H / \u222b_C measured               {measured_ratio:.2f}")
print(f"m_C / m_H, i.e. equipartition      {mass_ratio:.2f}")
print(f"m_C / m_H x T_H / T_C              {mass_ratio * ratio:.2f}  <- what is seen")
print()
print(
    "Welch against the direct mean-square velocity is the Parseval check on the "
    "estimator itself:\n"
    f"  C {parseval('C'):+.1f}%   H {parseval('H'):+.1f}%"
    "  -- so the absolute normalisation is sound and the\n"
    "  sum-rule failure is in the trajectory, not in stage B."
)

system kinetic temperature (3N-3 dof, COM removed): 20.21 K
the file's own frame-0 `temperature` info field:     25.22 K

species  atoms   mass    <|v|²> direct   ∫ tr P dE (Welch)   3k_BT_sys/m   T_species
   C      24   12.011        3.2907           3.2354        4.1968     15.85 K
   H      24    1.008       58.7214          58.9202       50.0078     23.73 K

T_H / T_C                          1.498
∫_H / ∫_C measured               18.21
m_C / m_H, i.e. equipartition      11.92
m_C / m_H x T_H / T_C              17.84  <- what is seen

Welch against the direct mean-square velocity is the Parseval check on the estimator itself:
  C -1.7%   H +0.3%  -- so the absolute normalisation is sound and the
  sum-rule failure is in the trajectory, not in stage B.


**T_H / T_C = 1.50, stationary over the whole run.** The split is the same in the first
100 frames as in the last (24.2 K against 23.7 K for hydrogen, 15.4 K against 15.8 K
for carbon), so it is not an equilibration transient that more sampling would cure. Two
readings, and this repository cannot distinguish them from one trajectory:

- **A frozen draw, again.** This is §3's argument in species-resolved form. In a
  near-harmonic molecular crystal the stiff intramolecular modes exchange energy with
  the soft lattice modes very slowly, so whatever the initial velocity assignment put
  into the C–H stretches stays there. Hydrogen dominates those modes, carbon dominates
  the lattice modes, and the two sub-systems equilibrate on a timescale much longer than
  100 ps. That is the same physics as the argon result and it is *why* one NVE
  trajectory cannot carry an honest error bar.
- **A timestep artefact.** A 1 fs step resolves a 380 meV C–H stretch with about
  eleven points per period, which is marginal for velocity Verlet. Systematic energy
  drift into the fastest modes would look like exactly this.

Either way it matters for INS, because the hydrogen projection is the spectrum — the
bound cross-section per unit mass is 81.4 barn/amu for ¹H against 0.46 for ¹²C, a factor
of 176 — and a hydrogen sub-system running 50% hot is a 50% error in the Debye-Waller
exponent before any lineshape has been computed.

`check_sum_rule` is doing its job here. It is the only thing in the pipeline that
noticed.


### 9.3 What the estimator sweep can still show

The spectrum below 20.68 meV is not the external-mode DOS of benzene and should not be
read as one. It is, however, a perfectly good real-data test bed for the estimator: the
question "does the answer move when I change the segment length?" does not need the
answer to be physically correct.

The three sliders match what the Lennard-Jones sections offer. Reported alongside each
is ⟨E⟩ of the **hydrogen** projection over the 0–20.68 meV grid — a stability metric,
not a physical first moment.

The ±1σ values printed are the Welch inter-segment spread. Read §3 before believing
them.


In [19]:
BZ_LENGTH_SEGMENT = int(np.searchsorted(BZ_SEGMENTS, 128))  # available at every length

figure = go.Figure()
for length_index in range(len(BZ_LENGTHS)):
    figure.add_trace(
        go.Scatter(
            x=BZ_E,
            y=np.round(bz["benzene_grid_total"][length_index, BZ_LENGTH_SEGMENT], 5),
            name="total pDOS, all 48 atoms",
            line={"color": HYDROGEN, "width": 1.8},
            visible=length_index == len(BZ_LENGTHS) - 1,
            showlegend=False,
        )
    )

steps = []
for length_index, frames in enumerate(BZ_LENGTHS):
    visible = [index == length_index for index in range(len(BZ_LENGTHS))]
    m1 = bz["benzene_grid_m1"][length_index, BZ_LENGTH_SEGMENT]
    s1 = bz["benzene_grid_s1"][length_index, BZ_LENGTH_SEGMENT]
    n_segments = int(bz["benzene_grid_n_segments"][length_index, BZ_LENGTH_SEGMENT])
    steps.append(
        {
            "label": f"{frames * BZ_DT:.1f}",
            "method": "update",
            "args": [
                {"visible": visible},
                {
                    "title.text": (
                        f"Solid benzene, first {frames * BZ_DT:.1f} of 100.1 ps "
                        f"({n_segments} Welch segments of "
                        f"{BZ_SEGMENTS[BZ_LENGTH_SEGMENT]} "
                        f"frames): ⟨E⟩<sub>H</sub> = {m1:.3f} ± {s1:.3f} meV"
                    )
                },
            ],
        }
    )

figure.update_layout(
    **LAYOUT,
    title=steps[-1]["args"][1]["title.text"],
    xaxis_title="energy transfer (meV)",
    yaxis_title="tr P / 3 summed over atoms (Å² ps⁻² meV⁻¹)",
    sliders=[
        {
            "active": len(steps) - 1,
            "currentvalue": {"prefix": "trajectory length: ", "suffix": " ps"},
            "pad": {"t": 50},
            "steps": steps,
        }
    ],
    height=500,
)
figure

In [20]:
figure = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    subplot_titles=(
        "hydrogen, per atom — the projection an INS spectrum is made of",
        "carbon, per atom — note the y axis is smaller by a factor of ~18",
    ),
)

BZ_DEFAULT_SEGMENT = int(np.searchsorted(BZ_SEGMENTS, 256))
for segment_index in range(len(BZ_SEGMENTS)):
    visible = segment_index == BZ_DEFAULT_SEGMENT
    for row, (species, colour) in enumerate((("H", HYDROGEN), ("C", CARBON)), start=1):
        figure.add_trace(
            go.Scatter(
                x=BZ_E,
                y=np.round(bz[f"benzene_grid_{species}"][-1, segment_index], 6),
                name=species,
                line={"color": colour, "width": 1.7},
                visible=visible,
                showlegend=False,
            ),
            row=row,
            col=1,
        )

steps = []
for segment_index, segment in enumerate(BZ_SEGMENTS):
    visible = [
        block == segment_index for block in range(len(BZ_SEGMENTS)) for _ in range(2)
    ]
    resolution = float(bz["benzene_grid_resolution"][-1, segment_index])
    n_segments = int(bz["benzene_grid_n_segments"][-1, segment_index])
    steps.append(
        {
            "label": f"{segment}",
            "method": "update",
            "args": [
                {"visible": visible},
                {
                    "title.text": (
                        f"Species-projected pDOS, full 100.1 ps, segment {segment} "
                        f"frames: {resolution:.3f} meV bins, {n_segments} averaged"
                    )
                },
            ],
        }
    )

figure.update_layout(
    **LAYOUT,
    title=steps[BZ_DEFAULT_SEGMENT]["args"][1]["title.text"],
    sliders=[
        {
            "active": BZ_DEFAULT_SEGMENT,
            "currentvalue": {"prefix": "Welch segment: ", "suffix": " frames"},
            "pad": {"t": 50},
            "steps": steps,
        }
    ],
    height=640,
)
figure.update_xaxes(title_text="energy transfer (meV)", row=2, col=1)
figure.update_yaxes(title_text="tr P / 3 (Å² ps⁻² meV⁻¹)", row=1, col=1)
figure.update_yaxes(title_text="tr P / 3 (Å² ps⁻² meV⁻¹)", row=2, col=1)
figure

In [21]:
figure = go.Figure()
BZ_WINDOW_SEGMENT = int(np.searchsorted(BZ_SEGMENTS, 256))
for window_index, name in enumerate(BZ_WINDOWS):
    figure.add_trace(
        go.Scatter(
            x=BZ_E,
            y=np.round(bz["benzene_window_H"][window_index, BZ_WINDOW_SEGMENT], 6),
            name=name,
            line={"color": CATEGORY[window_index], "width": 1.7},
            visible=window_index == 0,
            showlegend=False,
        )
    )

buttons = []
for window_index, name in enumerate(BZ_WINDOWS):
    m1 = bz["benzene_window_m1"][window_index, BZ_WINDOW_SEGMENT]
    m2 = bz["benzene_window_m2"][window_index, BZ_WINDOW_SEGMENT]
    buttons.append(
        {
            "label": name,
            "method": "update",
            "args": [
                {
                    "visible": [
                        index == window_index for index in range(len(BZ_WINDOWS))
                    ]
                },
                {
                    "title.text": (
                        f"Hydrogen pDOS, segment {BZ_SEGMENTS[BZ_WINDOW_SEGMENT]} "
                        f"frames, {name} window: ⟨E⟩ = {m1:.3f} meV, "
                        f"⟨E²⟩ = {m2:.2f} meV²"
                    )
                },
            ],
        }
    )

figure.update_layout(
    **LAYOUT,
    title=buttons[0]["args"][1]["title.text"],
    xaxis_title="energy transfer (meV)",
    yaxis_title="tr P / 3 per H atom (Å² ps⁻² meV⁻¹)",
    updatemenus=[
        {
            "type": "buttons",
            "direction": "right",
            "buttons": buttons,
            "x": 0.0,
            "xanchor": "left",
            "y": 1.13,
            "yanchor": "bottom",
            "showactive": True,
        }
    ],
    height=500,
)
figure

In [22]:
figure = go.Figure()
ESTIMATORS = (("welch", HYDROGEN, "solid"), ("vacf", CATEGORY[1], "dot"))
for segment_index in range(len(BZ_SEGMENTS)):
    for estimator_index, (name, colour, dash) in enumerate(ESTIMATORS):
        figure.add_trace(
            go.Scatter(
                x=BZ_E,
                y=np.round(
                    bz["benzene_estimator_H"][estimator_index, segment_index], 6
                ),
                name=f"{name} ({'segment' if name == 'welch' else 'max lag'})",
                line={"color": colour, "width": 1.7, "dash": dash},
                visible=segment_index == BZ_DEFAULT_SEGMENT,
                showlegend=segment_index == BZ_DEFAULT_SEGMENT,
            )
        )

steps = []
for segment_index, segment in enumerate(BZ_SEGMENTS):
    visible = [
        block == segment_index for block in range(len(BZ_SEGMENTS)) for _ in range(2)
    ]
    welch_m1 = bz["benzene_estimator_m1"][0, segment_index]
    vacf_m1 = bz["benzene_estimator_m1"][1, segment_index]
    steps.append(
        {
            "label": f"{segment}",
            "method": "update",
            "args": [
                {"visible": visible, "showlegend": visible},
                {
                    "title.text": (
                        f"Welch against VACF at {segment} frames: "
                        f"⟨E⟩<sub>H</sub> = {welch_m1:.3f} and {vacf_m1:.3f} meV, "
                        f"differing by {100 * abs(vacf_m1 / welch_m1 - 1):.2f}%"
                    )
                },
            ],
        }
    )

figure.update_layout(
    **LAYOUT,
    title=steps[BZ_DEFAULT_SEGMENT]["args"][1]["title.text"],
    xaxis_title="energy transfer (meV)",
    yaxis_title="tr P / 3 per H atom (Å² ps⁻² meV⁻¹)",
    sliders=[
        {
            "active": BZ_DEFAULT_SEGMENT,
            "currentvalue": {"prefix": "segment / max lag: ", "suffix": " frames"},
            "pad": {"t": 50},
            "steps": steps,
        }
    ],
    legend={"orientation": "h", "y": -0.3},
    height=520,
)
figure

In [23]:
print("Stability of the hydrogen projection across the estimator grid, full 100.1 ps.")
print("Nothing here is a physical number; the spread is the point.\n")
print("axis                              range of \u27e8E\u27e9 (meV)     spread")
for label, values in (
    ("trajectory length, 12.8-100.1 ps", bz["benzene_grid_m1"][:, BZ_LENGTH_SEGMENT]),
    ("Welch segment, 64-512 frames", bz["benzene_grid_m1"][-1]),
    ("window, 4 tapers at 256 frames", bz["benzene_window_m1"][:, BZ_WINDOW_SEGMENT]),
    ("overlap, 0-75% at 256 frames", bz["benzene_overlap_m1"]),
    ("estimator, welch vs vacf", bz["benzene_estimator_m1"][:, BZ_DEFAULT_SEGMENT]),
):
    finite = values[np.isfinite(values)]
    print(
        f"{label:33s}  {finite.min():.3f} - {finite.max():.3f}      "
        f"{100 * (finite.max() - finite.min()) / finite.mean():5.2f}%"
    )

print()
print("Overlap moves the quoted error bar, not the estimate \u2014 as in \u00a76.\n")
print(
    "overlap   segments   \u27e8E\u27e9 (meV)   quoted 1\u03c3"
    "   1\u03c3 as % of \u27e8E\u27e9"
)
for index, overlap in enumerate(BZ_OVERLAPS):
    m1 = bz["benzene_overlap_m1"][index]
    s1 = bz["benzene_overlap_s1"][index]
    n_segments = int(bz["benzene_overlap_n_segments"][index])
    print(
        f"  {overlap:>4.0%}      {n_segments:3d}       {m1:.3f}      {s1:.4f}"
        f"       {100 * s1 / m1:.2f}%"
    )

Stability of the hydrogen projection across the estimator grid, full 100.1 ps.
Nothing here is a physical number; the spread is the point.

axis                              range of ⟨E⟩ (meV)     spread
trajectory length, 12.8-100.1 ps   9.062 - 9.182       1.32%
Welch segment, 64-512 frames       9.170 - 9.225       0.60%
window, 4 tapers at 256 frames     9.173 - 9.204       0.33%
overlap, 0-75% at 256 frames       9.144 - 9.197       0.58%
estimator, welch vs vacf           9.183 - 9.211       0.30%

Overlap moves the quoted error bar, not the estimate — as in §6.

overlap   segments   ⟨E⟩ (meV)   quoted 1σ   1σ as % of ⟨E⟩
    0%        3       9.144      0.1147       1.25%
   25%        4       9.178      0.1155       1.26%
   50%        6       9.183      0.0860       0.94%
   75%       12       9.197      0.0596       0.65%


### 9.4 What this section establishes, and what it does not

**Does:**

- Stage A reads a third-party MACE-MP extxyz trajectory through `read_velocities`
  without special-casing: `momenta` and `masses` are present, so velocities come out in
  Å/ps and the dump interval is recoverable from `info['time']`.
- Stage B's absolute normalisation survives the round trip on real data. Welch
  reproduces the directly computed ⟨|v|²⟩ to −2% (C) and +0.3% (H).
- The estimator is stable on this trajectory: ⟨E⟩<sub>H</sub> moves by 1.32% across a
  factor of eight in trajectory length, 0.60% across a factor of eight in segment
  length, 0.33% across the four window functions, 0.58% across the four overlaps and
  0.30% between Welch and VACF.
- The species projection works and is checkable in absolute units, which is what turned
  up the non-equipartition.

**Does not:**

- It is **not** a validation. There is no harmonic reference for benzene here, so no
  claim about accuracy is supported by anything in this notebook.
- The spectrum is **not** benzene's external-mode DOS. 83% of the cell's degrees of
  freedom lie above the 20.68 meV Nyquist limit and their weight is folded into the
  visible band; decimating to 0.2 ps adds a further 65% (H) to the weight below
  10.33 meV, which is a direct measurement of the same effect one octave lower.
- The ±1σ values are inter-segment precision on one NVE run. §3 measured that
  quantity to be about 4.5× too small as an uncertainty on the answer for argon, and
  there is no second benzene seed with which to measure the equivalent factor. Treat
  every error bar in this section as a lower bound of unknown tightness.

**What would fix it.** Benzene is the canonical TOSCA benchmark precisely because its
intramolecular modes are sharp, well characterised and hydrogen-dominated, and none of
them are visible here. Reaching the 380 meV C–H stretch needs a dump interval of
`h / (2 x 380 meV) = 5.4 fs` or shorter — about twenty times finer than this file — and
a shorter integration timestep to go with it. `validate_sampling` will state that
requirement exactly if asked for `e_max = 400`; the useful habit is to ask it *before*
running the MD, not after.


## What to take away

1. **Length converges the estimate, not the answer.** Every seed flattens by 60-80 ps;
   they flatten at different values, and the spread between them does not shrink.
2. **The dominant uncertainty in these NVE runs is the initial draw**, at 4.5 times the
   Welch inter-segment error bar on both ⟨E⟩ and ⟨E²⟩ across ten seeds. The
   tolerances in `validation.md` are derived honestly from the quantity they name, but
   that quantity is a lower bound on the total uncertainty, and a rerun with a different
   seed could fail at 3σ with nothing wrong in the pipeline.
3. **Resolution is a trade, not an improvement.** 2048 frames is close to the best
   available compromise for this system; the moments are insensitive to it, which is
   reassuring for the test suite.
4. **`energy_resolution` is not the width of the peaks.** It is the bin spacing of one
   segment; the delivered width is that times the window's main lobe, 1.44 for Hann.
   Which taper is used barely matters at 2048 frames — but *not* tapering does, and it
   shows up in the tails and therefore in ⟨E²⟩, the more so the shorter the segment.
5. **Overlap moves the error bar and not the estimate**, so it should be read as a
   statement about bookkeeping.
6. **Temperature is a real systematic.** 10 K keeps anharmonicity to about the size of the
   Welch error; 19 K does not.
7. **The species projection is sound, and the two species converge at different rates** —
   the heavier species, with its denser low-energy modes, settles first.
8. **Agreement inside 1σ is not by itself evidence of a small error.** Extrapolating the
   temperature series of seed 0 to T → 0 suggests its excellent single-species agreement
   is a partial cancellation between a low frozen draw and anharmonic stiffening.

9. **A real trajectory is the estimator's easiest test and the sampling plan's hardest
   one.** Every estimator knob moves ⟨E⟩<sub>H</sub> of the benzene run by 1.3% or
   less, and the absolute normalisation checks out — but 83% of that system's degrees of
   freedom are above the Nyquist limit of its own dump interval, so the spectrum is
   wrong by far more than any estimator choice could make it. `e_max` has to be decided
   before the MD runs.
10. **The sum rule earns its place.** It is the only check in the pipeline that noticed
    the benzene run has hydrogen at 23.7 K and carbon at 15.8 K, stationary over 100 ps.
    Aliasing conserves total power, so the sum rule stays valid even when every feature
    is in the wrong place — which is what made it the useful diagnostic there.
